In [32]:
# %pip install pandas h5py numpy scipy matplotlib torch torchvision torchaudio -i https://pypi.tuna.tsinghua.edu.cn/simple --break-system-packages

In [33]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import h5py
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import scipy 

In [34]:
import scipy
data = scipy.io.loadmat('AmirSyn.mat')
Dc=data['Data']
Dn=data['Noisy']
print(f"Dimension of clean data: {Dc.shape}, Dimension of noisy data: {Dn.shape}")

干净数据维度: (500, 35, 40), 带噪数据维度: (500, 35, 40)


In [35]:
def calculate_snr(clean_data, denoised_data):
    # ensure the input is a 1D vector
    norm_clean = np.linalg.norm(clean_data)  # compute the square root of the sum of squares of all elements
    norm_diff = np.linalg.norm(clean_data - denoised_data)
    
    snr = 20 * np.log10(norm_clean / norm_diff)
    return snr

In [36]:
def patch3d(A, l1=12, l2=12, l3=12, s1=4, s2=2, s3=2):
    """
    Optimized patch3d function using stride tricks to improve efficiency
    """
    # Pad the array if necessary to ensure dimensions are divisible by patch sizes
    pad1 = (l1 - A.shape[0] % s1) % s1
    pad2 = (l2 - A.shape[1] % s2) % s2
    pad3 = (l3 - A.shape[2] % s3) % s3
    A_padded = np.pad(A, ((0, pad1), (0, pad2), (0, pad3)), mode='constant')

    # Get new dimensions
    n1, n2, n3 = A_padded.shape
    n1_patches = (n1 - l1) // s1 + 1
    n2_patches = (n2 - l2) // s2 + 1
    n3_patches = (n3 - l3) // s3 + 1

    # Generate the sliding window view
    shape = (n1_patches, n2_patches, n3_patches, l1, l2, l3)
    strides = (s1 * A_padded.strides[0], s2 * A_padded.strides[1], s3 * A_padded.strides[2]) + A_padded.strides
    patches = np.lib.stride_tricks.as_strided(A_padded, shape=shape, strides=strides)

    # Reshape patches to (num_patches, patch_size) format
    patches = patches.reshape(-1, l1 * l2 * l3)
    return patches


def patch3d_inv(X, n1, n2, n3, l1=12, l2=12, l3=12, s1=4, s2=2, s3=2):
    """
    Reconstruct 3D data from 1D patches with optimized inverse patching.

    INPUT
    X: Patches in (num_patches, patch_size) format
    n1, n2, n3: Original 3D data dimensions
    l1, l2, l3: Patch sizes along each dimension
    s1, s2, s3: Shifts between patches along each dimension

    OUTPUT
    A: Reconstructed 3D data
    """

    # Initialize the padded 3D array and mask
    pad1 = (l1 - n1 % s1) % s1
    pad2 = (l2 - n2 % s2) % s2
    pad3 = (l3 - n3 % s3) % s3
    A = np.zeros((n1 + pad1, n2 + pad2, n3 + pad3))
    mask = np.zeros_like(A)

    # Reshape 1D patches to the 3D patch size for easier handling
    X = X.reshape(-1, l1, l2, l3)

    # Calculate the number of patches along each dimension
    n1_patches = (A.shape[0] - l1) // s1 + 1
    n2_patches = (A.shape[1] - l2) // s2 + 1
    n3_patches = (A.shape[2] - l3) // s3 + 1

    # Iterate over each patch and place it into the appropriate location
    idx = 0
    for i in range(n1_patches):
        for j in range(n2_patches):
            for k in range(n3_patches):
                # Place the current patch in the reconstruction array and update mask
                A[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += X[idx]
                mask[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += 1
                idx += 1

    # Avoid division by zero by replacing zeros with ones in the mask before division
    mask[mask == 0] = 1
    A /= mask

    # Trim padding to match the original dimensions
    return A[:n1, :n2, :n3]

In [37]:
import scipy
[nz,nx,ny] =Dn.shape
w1 = 12
w2 = 12
w3 = 12
s1 = 4
s2 = 2
s3 = 2

X_noisy = patch3d(Dn,w1,w2,w3,s1,s2,s3)
# Convert to float32
X_noisy = X_noisy.astype(np.float32)

Pdata_noisy = torch.from_numpy(X_noisy)
print(Pdata_noisy.dtype)
print(f"Noisy patches shape: {X_noisy.shape}")
print(f"Noisy dtype: {X_noisy.dtype}")

# Data split: 80% training data, 20% validation data
train_size = int(len(Pdata_noisy) * 0.8)

train_noisy = Pdata_noisy[:train_size]
valid_noisy = Pdata_noisy[train_size:]

# Print the shapes of the training set and validation set
print(f"Train noisy shape: {train_noisy.shape}")
print(f"Valid noisy shape: {valid_noisy.shape}")

# Create TensorDataset
train_data = TensorDataset(train_noisy)
valid_data = TensorDataset(valid_noisy)

batch_size1 = 64

# Create DataLoader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size1, shuffle=False)

print(f"Train data loader batches: {len(train_loader)}")
print(f"Valid data loader batches: {len(valid_loader)}")
print(f"Valid noisy dtype: {valid_noisy.dtype}")

torch.float32
Noisy patches shape: (23985, 1728)
Noisy dtype: float32
Train noisy shape: torch.Size([19188, 1728])
Valid noisy shape: torch.Size([4797, 1728])
Train data loader batches: 300
Valid data loader batches: 75
Valid noisy dtype: torch.float32


In [38]:
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU(0.1, inplace=True)
        self.norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

In [39]:
class SparseAttention(nn.Module):
    def __init__(self, dim, num_heads=4, window_size=2):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)

    

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale

        block_size = self.window_size * 2
        num_blocks = max(1, N // block_size)  
        mask = torch.zeros(N, N, device=x.device)
        for i in range(num_blocks):
            start = i * block_size
            end = min(start + block_size, N)  # Prevent out-of-bounds errors
            mask[start:end, start:end] = 1
        mask = mask.unsqueeze(0).unsqueeze(0).bool()

        attn = attn.masked_fill(~mask, -1e9)
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class Sparse_PatchAE(nn.Module):
    def __init__(self, input_dim, embed_dim=256, depth=1, num_heads=4, window_size=2):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim

        self.embed = nn.Linear(input_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 2, embed_dim) * 0.02)

        self.num_heads = num_heads
        self.window_size = window_size

        self.blocks = nn.ModuleList([
            self._make_block(embed_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Linear(embed_dim, input_dim)
        

    def _make_block(self, dim):
        return nn.ModuleList([
            nn.LayerNorm(dim),
            SparseAttention(dim, self.num_heads, self.window_size),
            nn.LayerNorm(dim),
            nn.Sequential(
                nn.Linear(dim, dim * 4),
                nn.GELU(),
                nn.Linear(dim * 4, dim)
            )
        ])

    def forward(self, x):
        B = x.shape[0]
        x = self.embed(x).unsqueeze(1)  # [B, 1, embed_dim]

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed

        for norm1, attn, norm2, mlp in self.blocks:
            x = x + attn(norm1(x))
            x = x + mlp(norm2(x))

        x = self.norm(x)
        patch_token = x[:, 1, :]
        recon = self.decoder(patch_token)
        return recon


In [40]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()
        self.fcb1 = FCB(input_size, 512, dropout)
        self.Sparse1 = Sparse_PatchAE(512, 512, depth=4, window_size=1)  

        self.fcb2 = FCB(512, 128, dropout)
        self.Sparse2 = Sparse_PatchAE(128, 128, depth=4, window_size=1)

        self.fcb3 = FCB(128, 64, dropout)
        self.Sparse3 = Sparse_PatchAE(64, 64, depth=4, window_size=1)

        self.fcb4 = FCB(64, 32, dropout)
        self.Sparse4 = Sparse_PatchAE(32, 32, depth=4, window_size=1)

        self.fcb5 = FCB(32, 8, dropout)
        self.Sparse5 = Sparse_PatchAE(8, 8, depth=4, window_size=1)

    def forward(self, x):
        x1 = self.fcb1(x)
        y1 = self.Sparse1(x1)
        x2 = self.fcb2(x1)
        y2 = self.Sparse2(x2)
        x3 = self.fcb3(x2)
        y3 = self.Sparse3(x3)
        x4 = self.fcb4(x3)
        y4 = self.Sparse4(x4)
        x5 = self.fcb5(x4)
        y5 = self.Sparse5(x5)
        return x1, x2, x3, x4, x5, y1, y2, y3, y4, y5

In [41]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.fcb5 = FCB(8 + 8, 32, dropout)
        self.fcb4 = FCB(32 + 32, 64, dropout)
        self.fcb3 = FCB(64 + 64, 128, dropout)
        self.fcb2 = FCB(128 + 128, 512, dropout)
        self.fcb1 = FCB(512 + 512, output_size, dropout)

    def forward(self, x1, x2, x3, x4, x5, y1, y2, y3, y4, y5):
        y51 = torch.cat([x5, y5], dim=1)
        y41 = self.fcb5(y51)
        y41 = torch.cat([y41, y4], dim=1)
        y31 = self.fcb4(y41)
        y31 = torch.cat([y31, y3], dim=1)
        y21 = self.fcb3(y31)
        y21 = torch.cat([y21, y2], dim=1)
        y11 = self.fcb2(y21)
        y11 = torch.cat([y11, y1], dim=1)
        output = self.fcb1(y11)
        return output

In [42]:
class AutoCoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_size, dropout)
        self.decoder = Decoder(output_size, dropout)

    def forward(self, x):
        x1, x2, x3, x4, x5, y1, y2, y3, y4, y5 = self.encoder(x)
        output = self.decoder(x1, x2, x3, x4, x5, y1, y2, y3, y4, y5)
        return output

In [43]:
def huber_loss(pred, target, delta=1.35):
    error = pred - target
    abs_error = torch.abs(error)
    quadratic = torch.clamp(abs_error, max=delta)
    linear = abs_error - quadratic
    loss = 0.5 * quadratic ** 2 + delta * linear
    return loss.mean()


In [44]:
#======== Model initialization ========
w1, w2, w3 = 12, 12, 12
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoCoder(input_size=w1*w2*w3, output_size=w1*w2*w3, dropout=0.1).to(device)
# Force align final Decoder layer dims (3D flattened length)
model.decoder.fcb1 = torch.nn.Linear(1024, w1*w2*w3).to(device) 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# ======= Training loop =======
es_thres = 5
es_cnt = 0
prev_valid_loss = float('inf')
loss_train = []
loss_valid = []

num_epochs_stage1 = 50
# Random masking ratio
mask_ratio1 = 0.3 
mask_ratio2 = 0.1 

print("Start Stage 1 training: first 50 epochs")
for epoch in range(num_epochs_stage1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        noisy_patch = batch[0].to(device)  
        B = noisy_patch.shape[0]
        
        # Generate a [0, 1) uniform random number matrix with the same shape as the input
        rand_tensor = torch.rand_like(noisy_patch)
        
        # Generate boolean mask: set positions smaller than mask_ratio to True (pixels to be masked)
        random_mask_2d = rand_tensor < mask_ratio1
        
        # Construct input: invert the mask positions bitwise (~) to set masked positions to 0 and keep original values for others
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch
        
        optimizer.zero_grad()
        outputs = model(input_patch)
        
        # -------- Loss calculation: only focus on masked scatter points --------
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.35)
        loss = rec_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * input_patch.size(0)

    train_loss = train_loss / len(train_noisy)
    loss_train.append(train_loss)

    # The validation set must also use the same Random Masking logic for evaluation
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for batch in valid_loader:
            valid_patch = batch[0].to(device)
            B_val = valid_patch.shape[0]
            
            rand_tensor_val = torch.rand_like(valid_patch)
            random_mask_2d_val = rand_tensor_val < mask_ratio2
            
            # Construct validation set input
            input_val_patch = valid_patch * (~random_mask_2d_val)
            
            outputs = model(input_val_patch)
            
            # Calculate validation loss
            rec_loss_val = huber_loss(outputs[random_mask_2d_val], valid_patch[random_mask_2d_val], delta=1.35)
            val_loss = rec_loss_val
            
            valid_loss += val_loss.item() * valid_patch.size(0)

    valid_loss = valid_loss / len(valid_noisy)
    loss_valid.append(valid_loss)

    print(f"stage1 - Epoch[{epoch+1}/{num_epochs_stage1}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

    # Saving best model
    if valid_loss < prev_valid_loss:
        prev_valid_loss = valid_loss
        torch.save(model.state_dict(), 'Amirsynbest_model12,12,12,50_stage1.pth')
        print("The optimal model of Stage 1 has been saved")

#Stage 2: Load optimal parameters + continue training for another 50 epochs.
print("\nStart Stage 2 training: load the optimal model and train for another 50 epochs")
# Reset the best loss value.
prev_valid_loss = float('inf')
# Load the best model from Stage 1.
model.load_state_dict(torch.load('Amirsynbest_model12,12,12,50_stage1.pth'))
model.to(device)
# Rebind parameters to the optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs_stage2 = 50

for epoch in range(num_epochs_stage2):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        noisy_patch = batch[0].to(device)  
        B = noisy_patch.shape[0]
        
        rand_tensor = torch.rand_like(noisy_patch)
        random_mask_2d = rand_tensor < mask_ratio1
        input_patch = noisy_patch * (~random_mask_2d)
        target_patch = noisy_patch
        
        optimizer.zero_grad()
        outputs = model(input_patch)
        rec_loss = huber_loss(outputs[random_mask_2d], target_patch[random_mask_2d], delta=1.35)
        loss = rec_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * input_patch.size(0)

    train_loss = train_loss / len(train_noisy)
    loss_train.append(train_loss)

    #Validation phase
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for batch in valid_loader:
            valid_patch = batch[0].to(device)
            B_val = valid_patch.shape[0]
            
            rand_tensor_val = torch.rand_like(valid_patch)
            random_mask_2d_val = rand_tensor_val < mask_ratio2
            input_val_patch = valid_patch * (~random_mask_2d_val)
            
            outputs = model(input_val_patch)
            rec_loss_val = huber_loss(outputs[random_mask_2d_val], valid_patch[random_mask_2d_val], delta=1.35)
            val_loss = rec_loss_val
            valid_loss += val_loss.item() * valid_patch.size(0)

    valid_loss = valid_loss / len(valid_noisy)
    loss_valid.append(valid_loss)

    print(f"stage2 - Epoch[{epoch+1}/{num_epochs_stage2}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

    # Saving best model
    if valid_loss < prev_valid_loss:
        prev_valid_loss = valid_loss
        torch.save(model.state_dict(), 'Amirsynbest_model12,12,12,50_final.pth')
        print("The optimal model of Stage 2 has been saved")

print("\n All training procedures completed! The final optimal model is saved as Amirsynbest_model12,12,12,50_final.pth")

========== 开始第一阶段训练：前15个epoch ==========
阶段1 - Epoch[1/50], Train Loss: 0.1336, Valid Loss: 0.0930
✅ 阶段1最优模型已保存
阶段1 - Epoch[2/50], Train Loss: 0.1118, Valid Loss: 0.0915
✅ 阶段1最优模型已保存
阶段1 - Epoch[3/50], Train Loss: 0.1044, Valid Loss: 0.0905
✅ 阶段1最优模型已保存
阶段1 - Epoch[4/50], Train Loss: 0.0990, Valid Loss: 0.0901
✅ 阶段1最优模型已保存
阶段1 - Epoch[5/50], Train Loss: 0.0965, Valid Loss: 0.0895
✅ 阶段1最优模型已保存
阶段1 - Epoch[6/50], Train Loss: 0.0952, Valid Loss: 0.0894
✅ 阶段1最优模型已保存
阶段1 - Epoch[7/50], Train Loss: 0.0943, Valid Loss: 0.0890
✅ 阶段1最优模型已保存
阶段1 - Epoch[8/50], Train Loss: 0.0937, Valid Loss: 0.0894
阶段1 - Epoch[9/50], Train Loss: 0.0932, Valid Loss: 0.0890
阶段1 - Epoch[10/50], Train Loss: 0.0928, Valid Loss: 0.0892
阶段1 - Epoch[11/50], Train Loss: 0.0923, Valid Loss: 0.0892
阶段1 - Epoch[12/50], Train Loss: 0.0920, Valid Loss: 0.0890
阶段1 - Epoch[13/50], Train Loss: 0.0917, Valid Loss: 0.0889
✅ 阶段1最优模型已保存
阶段1 - Epoch[14/50], Train Loss: 0.0915, Valid Loss: 0.0892
阶段1 - Epoch[15/50], Train Loss: 0.0913